(mmm_control_dimensionality)=
# Control Dimensionality and ROAS in Media Mix Models

How many control variables should an MMM include? Adding neutral controls (variables that do not confound media effects) should not bias total ROAS, but it can change posterior uncertainty. [To select or not to select](https://arxiv.org/abs/2606.22850) (Section 5.4) shows that under independent Normal priors, expanding the control set can *inflate* implied prior $R^2$ and degrade treatment-effect inference, while a split R2D2 prior keeps the variance budget stable.

This notebook translates that experiment into a media mix modelling setting:

- **Treatment** $z$ becomes media spend (adstock + saturation).
- **Treatment effect** $\alpha$ becomes **total ROAS**, tracked via incrementality.
- **Covariates** $X$ become $K$ standardized controls, independent of spend.
- **$M_{\text{base}}$** is an MMM with `control_columns=None` ($K=0$).
- **$M_{\text{full}}$** fits the first $K$ controls from a fixed pool of $K_{\max}=100$.

We compare two control-coefficient priors that leave media priors untouched:

1. **Normal** — independent $\mathcal{N}(0, 0.5)$ coefficients (variance grows with $K$).
2. **Split R2D2** — a Dirichlet-style simplex over controls with a fixed $R^2$ budget.

The joint R2D2 prior (shrinking media and controls together) is out of scope; the paper shows it is badly biased for treatment effects.


## Prepare notebook

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pymc.dims as pmd
import pytensor.tensor as pt
import pytensor.xtensor as ptx
import seaborn as sns
import xarray as xr
from pymc_extras.prior import Prior

from pymc_marketing.mmm import GeometricAdstock, LogisticSaturation
from pymc_marketing.mmm.mmm import MMM
from pymc_marketing.mmm.transformers import geometric_adstock, logistic_saturation
from pymc_marketing.special_priors import SpecialPrior

az.style.use("arviz-darkgrid")
plt.rcParams["figure.figsize"] = [10, 6]
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.facecolor"] = "white"

%config InlineBackend.figure_format = "retina"

In [ ]:
seed = sum(map(ord, "control-dim"))
rng = np.random.default_rng(seed)

# DGP and experiment knobs
N_DATES = 156
N_CHANNELS = 5
K_MAX = 100
L_MAX = 6
CHOL_ETA = 3
TARGET_R2 = 0.9
MEDIA_SHARE_OF_SALES = 0.15  # 15% of total sales; typical MMM range is 10-20%
INTERCEPT_TRUE = 1.0
GAMMA_CONTROL_MAX = 0.1
K_GRID = (0, 5, 10, 25, 50, 100)
PRIOR_LABELS = ("normal", "r2d2")

# Sampler settings (raise for publication-quality posteriors)
CHAINS = 4
TUNE = 1_000
DRAWS = 500
N_REPS = 1

## Data-generating process

We generate **one** dataset and hold it fixed across all fits, mirroring the paper's nested subset design.

- **Media spend** follows the LKJ Cholesky generative model from {ref}`mmm_data_generator` (correlated channels, trends, non-negativity via softplus).
- **Controls** are $K_{\max}=100$ i.i.d. standard-normal columns, independent of spend. Coefficients decrease linearly from $c_0$ (largest) to $c_{K-1}$ (near zero), so early controls carry most predictive power and the tail is noise.
- **Intercept** is fixed at a constant baseline so level shares are well defined.
- **Media** passes through `geometric_adstock` and `logistic_saturation` with fixed $\alpha$ and $\lambda$; $\beta$ is solved so media accounts for ${\sim}15\%$ of total sales (within the typical $10$–$20\%$ MMM range).
- **Noise** $\sigma$ is solved to hit true $R^2 = 0.9$ (high-signal regime: media plus controls explain most variation).

Ground-truth total ROAS is $\sum_t \text{media}_t / \sum_t \text{spend}_t$, a single number identical for every $K$.


In [ ]:
def _channel_names(n_channels: int = N_CHANNELS) -> list[str]:
    return [f"x{i}" for i in range(n_channels)]


def _control_names(k: int) -> list[str]:
    return [f"c{i}" for i in range(k)]


def _true_control_coefficients(
    k_max: int = K_MAX,
    gamma_max: float = GAMMA_CONTROL_MAX,
) -> np.ndarray:
    """Linearly decreasing coefficients: first controls matter most."""
    weights = np.linspace(1.0, 0.0, k_max)
    return gamma_max * weights


def generate_media_spend(rng: np.random.Generator) -> pd.DataFrame:
    channels = _channel_names()
    date_range = pd.date_range(start="2020-01-01", freq="W-MON", periods=N_DATES)
    coords = {"channel": channels, "date": date_range}
    t = np.arange(N_DATES) / N_DATES

    with pm.Model(coords=coords) as covariates_model:
        t_data = pm.Data("t", t, dims=("date",))
        L, _, _ = pm.LKJCholeskyCov(
            "L",
            n=len(coords["channel"]),
            eta=CHOL_ETA,
            sd_dist=pm.Exponential.dist(lam=1 / 3),
        )
        a = pm.Normal("a", mu=0, sigma=1, dims="channel")
        b = pm.Normal("b", mu=0, sigma=1, dims="channel")
        mu = pm.Deterministic("mu", a + b * t_data[..., None], dims=("date", "channel"))
        x_raw = pm.MvNormal("x_raw", mu=mu, chol=L, dims=("date", "channel"))
        pm.Deterministic("x", pt.softplus(x_raw), dims=("date", "channel"))

    spend = pm.draw(covariates_model.x, draws=1, random_seed=rng)
    spend_arr = spend.squeeze(0) if spend.ndim == 3 else spend
    spend_df = pd.DataFrame(spend_arr, columns=channels)
    spend_df["date"] = date_range
    return spend_df


def _media_base_contribution(
    spend_df: pd.DataFrame,
    adstock_alpha_true: np.ndarray,
    saturation_lam_true: np.ndarray,
) -> np.ndarray:
    channels = _channel_names()
    spend_xt = ptx.as_xtensor(spend_df[channels].to_numpy(), dims=("date", "channel"))
    alpha_xt = ptx.as_xtensor(adstock_alpha_true, dims=("channel",))
    lam_xt = ptx.as_xtensor(saturation_lam_true, dims=("channel",))
    adstocked = geometric_adstock(spend_xt, alpha=alpha_xt, l_max=L_MAX, dim="date")
    saturated = logistic_saturation(adstocked, lam=lam_xt)
    return saturated.sum("channel").eval()


def _solve_beta_for_media_share(
    base_media: np.ndarray,
    control_contribution: np.ndarray,
    intercept: float,
    target_share: float,
) -> float:
    """Solve beta so sum(media) / sum(intercept + media + control) == target_share."""

    def level_share(beta: float) -> float:
        media = beta * base_media
        mu = intercept + media + control_contribution
        return media.sum() / mu.sum()

    lo, hi = 1e-12, 1e8
    if level_share(hi) < target_share:
        raise ValueError(
            "Cannot reach target media share; increase intercept or base media scale"
        )
    for _ in range(80):
        mid = (lo + hi) / 2
        if level_share(mid) > target_share:
            hi = mid
        else:
            lo = mid
    return (lo + hi) / 2


def simulate(seed: int) -> tuple[pd.DataFrame, dict]:
    local_rng = np.random.default_rng(seed)
    spend_df = generate_media_spend(local_rng)
    controls = local_rng.standard_normal((N_DATES, K_MAX))
    controls = (controls - controls.mean(axis=0)) / controls.std(axis=0, ddof=0)
    control_names = _control_names(K_MAX)
    controls_df = pd.DataFrame(controls, columns=control_names)
    gamma_control_true = _true_control_coefficients()

    adstock_alpha_true = pm.draw(
        pm.Beta.dist(alpha=1, beta=3), draws=N_CHANNELS, random_seed=local_rng
    )
    saturation_lam_true = pm.draw(
        pm.Gamma.dist(alpha=2, beta=1), draws=N_CHANNELS, random_seed=local_rng
    )
    base_media = _media_base_contribution(
        spend_df, adstock_alpha_true, saturation_lam_true
    )
    control_contribution = controls @ gamma_control_true
    beta_media = _solve_beta_for_media_share(
        base_media,
        control_contribution,
        INTERCEPT_TRUE,
        MEDIA_SHARE_OF_SALES,
    )
    saturation_beta_true = np.full(N_CHANNELS, beta_media)

    media_contribution = base_media * beta_media
    mu = INTERCEPT_TRUE + media_contribution + control_contribution
    var_mu = float(np.var(mu, ddof=0))
    sigma_true = np.sqrt(var_mu * (1 - TARGET_R2) / TARGET_R2)
    y = mu + local_rng.normal(0, sigma_true, size=N_DATES)

    data = pd.concat([spend_df, controls_df], axis=1)
    data["y"] = y

    total_media = float(media_contribution.sum())
    total_spend = float(spend_df[_channel_names()].to_numpy().sum())
    truth = {
        "true_total_roas": total_media / total_spend,
        "sigma_true": sigma_true,
        "var_media": float(np.var(media_contribution, ddof=0)),
        "var_control": float(np.var(control_contribution, ddof=0)),
        "var_mu": var_mu,
        "media_share_of_sales": float(total_media / y.sum()),
        "media_share_of_mu": float(total_media / mu.sum()),
        "adstock_alpha_true": adstock_alpha_true,
        "saturation_lam_true": saturation_lam_true,
        "saturation_beta_true": saturation_beta_true,
        "gamma_control_true": gamma_control_true,
    }
    return data, truth


data, truth = simulate(seed)
print(f"True total ROAS: {truth['true_total_roas']:.4f}")
print(
    f"True R²: {TARGET_R2:.2f}, media share of sales: {truth['media_share_of_sales']:.1%} "
    f"(target {MEDIA_SHARE_OF_SALES:.0%})"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
channels = _channel_names()
melted = data.melt(
    id_vars="date", value_vars=channels, var_name="channel", value_name="spend"
)
sns.lineplot(data=melted, x="date", y="spend", hue="channel", ax=axes[0], legend=False)
axes[0].set_title("Synthetic media spend")
axes[0].set_xlabel("date")

sample_controls = data[_control_names(10)]
axes[1].imshow(sample_controls.T, aspect="auto", cmap="coolwarm", vmin=-2, vmax=2)
axes[1].set_title("First 10 standardized controls")
axes[1].set_xlabel("week")
axes[1].set_ylabel("control index")
plt.tight_layout()

## Control-coefficient priors

Both priors plug into `model_config["gamma_control"]`, leaving media priors unchanged.

### Normal prior

The default MMM uses independent Normal coefficients. Because `sigma` does not shrink with $K$, the implied prior on $\operatorname{Var}(\mu)$ grows linearly in $K$ while the residual scale prior stays fixed, so implied $R^2$ concentrates near 1.

### Split R2D2 prior

We implement `R2D2Prior` as a `SpecialPrior` using `pymc.dims` primitives (the `pymc_extras.distributions.R2D2M2CP` helper targets plain PyMC graphs and would need non-trivial wrapping for dims-based MMMs).

**Caveats:**

- The reference Stan model conditions the scale on the sampled residual $\sigma$; here we use a plug-in `sigma_ref` because `gamma_control` is created before the likelihood $\sigma$ in `mmm.py`.
- R2D2 assumes standardized covariates; our controls are standardized by construction (the MMM does not scale controls).


In [ ]:
class R2D2Prior(SpecialPrior):
    """Split R2D2 prior for control coefficients."""

    def _checks(self) -> None:
        allowed = {"r2_mean", "r2_prec", "concentration", "sigma_ref"}
        if set(self.parameters.keys()) != allowed:
            raise ValueError(f"Parameters must be {sorted(allowed)}")

    def create_variable(self, name: str, xdist: bool = False):
        """Create the R2D2 control coefficient vector."""
        if not xdist:
            raise NotImplementedError(f"{self!r} only supports xdist=True")
        r2_mean = self.parameters["r2_mean"]
        r2_prec = self.parameters["r2_prec"]
        concentration = self.parameters["concentration"]
        sigma_ref = self.parameters["sigma_ref"]
        r2 = pmd.Beta(
            f"{name}_R2",
            alpha=r2_mean * r2_prec,
            beta=(1 - r2_mean) * r2_prec,
        )
        g = pmd.Gamma(f"{name}_gamma", alpha=concentration, beta=1, dims=self.dims)
        psi = g / g.sum("control")
        tau2 = sigma_ref**2 * r2 / (1 - r2)
        z = pmd.Normal(f"{name}_z", mu=0, sigma=1, dims=self.dims)
        return pmd.Deterministic(name, z * pmd.math.sqrt(tau2 * psi))


def control_prior(kind: str):
    if kind == "normal":
        return Prior("Normal", mu=0, sigma=0.5, dims=("control",))
    if kind == "r2d2":
        return R2D2Prior(
            dims=("control",),
            r2_mean=1 / 3,
            r2_prec=3,
            concentration=1,
            sigma_ref=1.0,
        )
    raise ValueError(kind)


_ = R2D2Prior(
    dims=("control",),
    r2_mean=1 / 3,
    r2_prec=3,
    concentration=1,
    sigma_ref=1.0,
).sample_prior(
    coords={"control": _control_names(5)},
    name="gamma_control",
    draws=20,
    random_seed=rng,
)
print("R2D2Prior builds and samples via sample_prior.")

## Implied prior $R^2$ vs control dimensionality

Figure 4 (left) in the paper plots implied prior $R^2$ as $p$ grows. We reproduce the analogue from **prior predictive draws only** (no MCMC):

$$R^2 = \frac{\operatorname{Var}(\mu)}{\operatorname{Var}(\mu) + \sigma^2}$$

where $\mu$ is the prior mean function (intercept + media + controls). The Normal prior's implied $R^2$ should concentrate at 1 as $K$ grows; R2D2 stays stable.


In [ ]:
def implied_prior_r2(k: int, prior_kind: str, *, draws: int = 300) -> xr.DataArray:
    channels = _channel_names()
    control_cols = None if k == 0 else _control_names(k)
    date_range = pd.date_range(start="2020-01-01", freq="W-MON", periods=N_DATES)
    local_rng = np.random.default_rng(seed + k)

    spend = local_rng.uniform(1, 10, size=(N_DATES, N_CHANNELS))
    X = pd.DataFrame(spend, columns=channels)
    X["date"] = date_range
    if control_cols is not None:
        controls = local_rng.standard_normal((N_DATES, k))
        for idx, col in enumerate(control_cols):
            X[col] = controls[:, idx]
    y = pd.Series(np.zeros(N_DATES))

    model_config = {
        "intercept": Prior("Normal", mu=0, sigma=1),
        "adstock_alpha": Prior("Beta", alpha=1, beta=3, dims="channel"),
        "saturation_lam": Prior("Gamma", alpha=2, beta=1, dims="channel"),
        "saturation_beta": Prior("HalfNormal", sigma=2, dims="channel"),
        "likelihood": Prior("Normal", sigma=Prior("HalfNormal", sigma=1)),
        "gamma_control": control_prior(prior_kind),
    }

    mmm = MMM(
        date_column="date",
        target_column="y",
        channel_columns=channels,
        control_columns=control_cols,
        adstock=GeometricAdstock(l_max=L_MAX),
        saturation=LogisticSaturation(),
        yearly_seasonality=None,
        model_config=model_config,
    )
    mmm.build_model(X, y)
    with mmm.model:
        prior_idata = pm.sample_prior_predictive(draws=draws, random_seed=seed + k)
    prior = prior_idata.prior

    mu_media = prior["channel_contribution"].sum("channel")
    mu_control = (
        prior["control_contribution"].sum("control")
        if "control_contribution" in prior
        else 0
    )
    mu = mu_media + mu_control + prior["intercept_contribution"]
    sigma = prior["y_sigma"]
    var_mu = mu.var(dim="date")
    return var_mu / (var_mu + sigma**2)


prior_r2_records = []
for k in K_GRID:
    if k == 0:
        continue
    for prior_kind in PRIOR_LABELS:
        r2 = implied_prior_r2(k, prior_kind)
        prior_r2_records.append(
            pd.DataFrame(
                {
                    "k": k,
                    "prior": prior_kind,
                    "r2": r2.values.ravel(),
                }
            )
        )
prior_r2_df = pd.concat(prior_r2_records, ignore_index=True)

g = sns.FacetGrid(
    prior_r2_df, col="prior", hue="k", col_wrap=1, height=3, sharex=True, sharey=True
)
g.map_dataframe(sns.kdeplot, x="r2", fill=True, alpha=0.35, linewidth=1)
g.add_legend(title="K")
g.set_axis_labels("implied prior R²", "")
g.fig.suptitle(
    "Implied prior R² grows with K under Normal controls, stays stable under R2D2",
    y=1.02,
)
plt.tight_layout()

## Single-dataset ROAS experiment

We fit nested subsets $K \in \{0, 5, 10, 25, 50, 100\}$ on the **same** target $y$. For $K=0$ we pass `control_columns=None` (not an empty list). Each $K>0$ is fit under both priors; $K=0$ is fit once.

Total ROAS is computed from posterior incrementality divided by total spend.


In [ ]:
def fit_for_k(data: pd.DataFrame, k: int, prior_kind: str, fit_seed: int):
    channels = _channel_names()
    control_cols = None if k == 0 else _control_names(k)
    model_config = {
        "intercept": Prior("Normal", mu=0, sigma=1),
        "adstock_alpha": Prior("Beta", alpha=1, beta=3, dims="channel"),
        "saturation_lam": Prior("Gamma", alpha=2, beta=1, dims="channel"),
        "saturation_beta": Prior("HalfNormal", sigma=2, dims="channel"),
        "likelihood": Prior("Normal", sigma=Prior("HalfNormal", sigma=1)),
        "gamma_control": control_prior(prior_kind),
    }

    mmm = MMM(
        date_column="date",
        target_column="y",
        channel_columns=channels,
        control_columns=control_cols,
        adstock=GeometricAdstock(l_max=L_MAX),
        saturation=LogisticSaturation(),
        yearly_seasonality=None,
        model_config=model_config,
        sampler_config={"nuts_sampler": "nutpie"},
    )

    feature_cols = ["date", *channels]
    if control_cols is not None:
        feature_cols.extend(control_cols)
    X = data[feature_cols]
    y = data["y"]

    mmm.fit(
        X,
        y,
        chains=CHAINS,
        tune=TUNE,
        draws=DRAWS,
        target_accept=0.9,
        random_seed=fit_seed,
    )

    incr = mmm.incrementality.compute_incremental_contribution(frequency="all_time")
    total_spend = mmm.data.get_channel_spend().sum()
    total_roas = incr.sum("channel") / total_spend
    return mmm, total_roas


fit_results: dict[tuple[int, str], xr.DataArray] = {}
fitted_mmms: dict[tuple[int, str], MMM] = {}
diagnostics: list[dict] = []

for k in K_GRID:
    priors = ("normal",) if k == 0 else PRIOR_LABELS
    for prior_kind in priors:
        mmm, total_roas = fit_for_k(data, k, prior_kind, seed + 10_000 + k)
        fit_results[(k, prior_kind)] = total_roas
        fitted_mmms[(k, prior_kind)] = mmm
        rhat = az.rhat(mmm.idata)
        diagnostics.append(
            {
                "k": k,
                "prior": prior_kind,
                "divergences": int(mmm.idata.sample_stats["diverging"].sum().values),
                "max_rhat": float(
                    np.nanmax([rhat[v].max().item() for v in rhat.data_vars])
                ),
            }
        )

diag_df = pd.DataFrame(diagnostics)
diag_df

In [ ]:
roas_frames = []
for (k, prior_kind), total_roas in fit_results.items():
    roas_frames.append(
        pd.DataFrame(
            {
                "k": k,
                "prior": prior_kind,
                "total_roas": total_roas.values.ravel(),
            }
        )
    )
roas_df = pd.concat(roas_frames, ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 5))
for prior_kind, sub in roas_df.groupby("prior"):
    for k, kk in sub.groupby("k"):
        sns.kdeplot(
            kk["total_roas"], ax=ax, label=f"{prior_kind}, K={k}", linewidth=1.2
        )

ax.axvline(truth["true_total_roas"], color="black", linestyle="--", label="true ROAS")
if (0, "normal") in fit_results:
    baseline = float(fit_results[(0, "normal")].mean())
    ax.axvline(baseline, color="gray", linestyle=":", label="K=0 posterior mean")

ax.set_xlabel("total ROAS")
ax.set_title("Total ROAS posterior densities across K (Figure 9 analogue)")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
true_media_frac = truth["media_share_of_mu"]

for prior_kind, ax in zip(PRIOR_LABELS, axes, strict=True):
    post = fitted_mmms[(100, prior_kind)].idata.posterior
    media = post["channel_contribution"].sum("date").sum("channel")
    control = post["control_contribution"].sum("date").sum("control")
    intercept = post["intercept_contribution"]
    if "date" in intercept.dims:
        intercept = intercept.sum("date")
    total = media + control + intercept
    media_frac = (media / total).values.ravel()
    control_frac = (control / total).values.ravel()
    ax.scatter(media_frac, control_frac, alpha=0.2, s=8)
    ax.axvline(true_media_frac, color="black", linestyle="--", label="true media share")
    ax.set_xlabel("posterior media share")
    ax.set_ylabel("posterior control share")
    ax.set_title(f"K=100, prior={prior_kind}")
    ax.legend()

plt.tight_layout()

## Repetition study

In [ ]:
def run_experiment(n_reps: int = N_REPS) -> pd.DataFrame:
    rows: list[dict] = []
    for rep in range(n_reps):
        if rep == 0 and n_reps == 1:
            rep_truth = truth
            for (k, prior_kind), total_roas in fit_results.items():
                q05, q50, q95 = np.quantile(total_roas.values, [0.05, 0.5, 0.95])
                rows.append(
                    {
                        "rep": rep,
                        "k": k,
                        "prior": prior_kind,
                        "true_roas": rep_truth["true_total_roas"],
                        "median": q50,
                        "interval_length": q95 - q05,
                        "covered": q05 <= rep_truth["true_total_roas"] <= q95,
                        "sq_error": (q50 - rep_truth["true_total_roas"]) ** 2,
                    }
                )
            continue

        rep_data, rep_truth = simulate(seed + rep)
        for k in K_GRID:
            priors = ("normal",) if k == 0 else PRIOR_LABELS
            for prior_kind in priors:
                _, total_roas = fit_for_k(
                    rep_data, k, prior_kind, seed + rep * 100_000 + k
                )
                q05, q50, q95 = np.quantile(total_roas.values, [0.05, 0.5, 0.95])
                rows.append(
                    {
                        "rep": rep,
                        "k": k,
                        "prior": prior_kind,
                        "true_roas": rep_truth["true_total_roas"],
                        "median": q50,
                        "interval_length": q95 - q05,
                        "covered": q05 <= rep_truth["true_total_roas"] <= q95,
                        "sq_error": (q50 - rep_truth["true_total_roas"]) ** 2,
                    }
                )
    return pd.DataFrame(rows)


summary = run_experiment(N_REPS)
agg = (
    summary.groupby(["k", "prior"])
    .agg(
        interval_length=("interval_length", "mean"),
        coverage=("covered", "mean"),
        rmse=("sq_error", lambda s: np.sqrt(s.mean())),
    )
    .reset_index()
)
agg

## Conclusion

On a fixed synthetic dataset with neutral controls, expanding $K$ under independent Normal priors inflates implied prior $R^2$ and can destabilize total ROAS posteriors even when MCMC diagnostics look fine. A split R2D2 prior on control coefficients keeps the variance budget stable while leaving media priors untouched.

Natural follow-ups: upstream `R2D2Prior` into `pymc_marketing.special_priors`, condition the R2D2 scale on the sampled likelihood $\sigma$, and extend the spend generator to geo-level MMMs.


In [ ]:
%load_ext watermark
%watermark -n -u -v -iv -w -p pymc_marketing,pytensor